# BRCA all omics benchmark
- benchmark for the brca dataset
- take
    - [x] mRNA
    - [x] CNA
    - [x] miRNA
    - [ ] DNA methylation data
- benchmark several prediction methods on them, using
    - [ ] early integration
    - [ ] late integration

In [4]:
import numpy as np

# Assuming X is your input matrix with shape (n_omics, n_samples, n_features)
# Example matrix for demonstration
X = np.array([
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]], # Omics 1
    [[10, 11, 12], [13, 14, 15], [16, 17, 18]], # Omics 2
    [[20, 21, 22], [23, 24, 25], [26, 27, 28]]
])

# Transpose the matrix to make n_samples the first dimension
X_transposed = np.transpose(X, (1, 0, 2))

# Reshape the matrix to combine n_features and n_omics into a single dimension
X_reshaped = X_transposed.reshape(X_transposed.shape[0], -1)

print("Original shape:", X.shape)
print("Transposed shape:", X_transposed.shape)
print("Reshaped shape:", X_reshaped.shape)

X, X_reshaped

Original shape: (3, 3, 3)
Transposed shape: (3, 3, 3)
Reshaped shape: (3, 9)


(array([[[ 1,  2,  3],
         [ 4,  5,  6],
         [ 7,  8,  9]],
 
        [[10, 11, 12],
         [13, 14, 15],
         [16, 17, 18]],
 
        [[20, 21, 22],
         [23, 24, 25],
         [26, 27, 28]]]),
 array([[ 1,  2,  3, 10, 11, 12, 20, 21, 22],
        [ 4,  5,  6, 13, 14, 15, 23, 24, 25],
        [ 7,  8,  9, 16, 17, 18, 26, 27, 28]]))

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variational_selection

In [3]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

In [4]:
# ensure that the omic channels are alined
assert mrna.columns[1:] == cna.columns[1:] == mirna.columns[1:]

In [10]:
mrna

sample,TCGA-AR-A0U3-01,TCGA-B6-A0IJ-01,TCGA-A2-A0D1-01,TCGA-BH-A0H0-01,TCGA-A8-A08G-01,TCGA-A7-A0DB-01,TCGA-B6-A0RE-01,TCGA-D8-A147-01,TCGA-AO-A0JC-01,TCGA-BH-A0BV-01,TCGA-AO-A12E-01,TCGA-AN-A049-01,TCGA-A8-A06R-01,TCGA-B6-A0RT-01,TCGA-AN-A0FL-01,TCGA-E2-A109-01,TCGA-BH-A0H5-01,TCGA-A8-A085-01,TCGA-BH-A0B7-01,TCGA-AN-A0AS-01,TCGA-A7-A0DA-01,TCGA-A8-A09Z-01,TCGA-D8-A13Y-01,TCGA-AR-A1AN-01,TCGA-A8-A06P-01,TCGA-BH-A0B3-01,TCGA-E2-A14Z-01,TCGA-E2-A10B-01,TCGA-AO-A12A-01,TCGA-E2-A152-01,TCGA-AO-A12F-01,TCGA-AO-A0JF-01,TCGA-A2-A04R-01,TCGA-BH-A0HY-01,TCGA-BH-A18I-01,TCGA-E2-A15I-01,…,TCGA-A8-A095-01,TCGA-A8-A06T-01,TCGA-BH-A0B9-01,TCGA-BH-A0HB-01,TCGA-B6-A0IH-01,TCGA-A2-A0EY-01,TCGA-BH-A0HW-01,TCGA-C8-A1HG-01,TCGA-AN-A0FV-01,TCGA-E2-A10E-01,TCGA-E2-A15H-01,TCGA-BH-A0H3-01,TCGA-AR-A1AS-01,TCGA-AN-A0FT-01,TCGA-A2-A04Y-01,TCGA-BH-A0DG-01,TCGA-AO-A03M-01,TCGA-AR-A1AI-01,TCGA-D8-A142-01,TCGA-B6-A0WV-01,TCGA-AN-A0XS-01,TCGA-A8-A06U-01,TCGA-BH-A0E2-01,TCGA-AO-A0JL-01,TCGA-A2-A0EM-01,TCGA-D8-A140-01,TCGA-AN-A0XV-01,TCGA-A8-A079-01,TCGA-BH-A0HX-01,TCGA-A8-A08H-01,TCGA-E2-A1B4-01,TCGA-BH-A1EV-01,TCGA-AN-A0FJ-01,TCGA-AN-A0FF-01,TCGA-E2-A158-01,TCGA-A8-A07W-01,TCGA-A8-A0AD-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ARHGEF10L""",8.8372,10.8905,11.1395,11.1699,10.3217,9.4012,10.0817,10.4024,9.2568,9.3338,9.8929,9.2288,9.3826,10.7764,9.331,9.8626,9.3931,9.935,9.1984,8.7282,10.7025,9.9907,8.7406,9.2716,10.2363,9.708,9.8046,9.8193,9.1645,9.8236,9.5565,9.2747,9.5425,9.34,9.7737,9.1406,…,9.6137,9.0278,10.0971,9.851,9.9487,9.5451,10.3648,9.2057,9.0805,9.4114,9.4848,9.4787,9.4665,9.1426,9.3774,8.8111,9.8006,11.2755,9.7079,9.5917,10.0705,9.0977,9.6913,10.4864,9.3572,10.0491,9.1558,9.5647,9.8004,9.3494,10.4367,9.0134,10.3856,10.5067,9.2199,9.4877,9.6338
"""HIF3A""",0.6512,1.697,1.9355,3.1817,1.4547,3.9239,1.729,4.7636,5.352,1.5453,4.4619,2.8901,2.9759,7.242,7.1398,0.944,4.3885,0.0,4.3177,0.0,6.7245,3.5046,2.2174,4.3498,3.4911,3.1799,2.9498,1.2765,3.8037,2.3097,2.0224,3.1812,1.4911,0.0,1.6494,3.0117,…,1.7647,1.0483,9.5132,1.5425,5.9683,1.3008,0.9186,2.6164,2.9125,2.0468,1.1422,3.8309,1.8881,1.167,3.6294,3.8834,3.8709,5.553,1.6438,0.5359,2.9143,1.3538,0.4777,2.7009,1.4728,2.7839,5.1924,0.7465,2.2669,4.5128,1.8654,0.4731,6.9051,3.4019,4.3358,0.7098,0.0
"""RNF17""",0.0,0.0,0.0,0.0,1.2933,0.0,0.0,0.4173,0.0,0.0,0.0,0.0,0.0,0.71,0.0,0.6923,1.0902,0.0,0.0,0.0,0.0,0.0,0.449,0.0,0.0,1.1055,0.0,0.0,0.0,0.0,1.2395,0.0,0.0,0.0,0.0,0.0,…,0.425,0.0,0.0,0.0,0.0,0.0,0.375,0.7934,0.0,0.0,0.0,0.7184,0.0,0.0,0.0,1.625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6783,0.0,0.0,1.9486,0.5218
"""RNF10""",12.3459,11.4899,13.0469,11.6484,12.154,11.9896,11.4039,11.4056,11.7543,11.9997,12.0904,12.0875,11.6885,11.3035,11.6591,11.9256,11.884,11.4895,11.4783,11.5278,12.0579,10.8292,11.0435,12.3459,11.7964,12.2457,12.3889,12.4981,12.2109,12.1153,11.5805,11.8001,11.9362,11.4961,11.82,12.0196,…,11.5408,12.4675,12.0289,12.4752,11.7441,11.5879,12.1031,11.5558,12.09,12.0519,12.3655,11.7907,12.2446,11.4574,12.0841,11.9622,12.3011,11.3922,11.8888,12.0036,11.918,11.7289,11.9049,11.3011,11.6602,12.1686,11.7876,12.1118,11.764,11.6758,12.428,11.7832,11.6644,12.2998,11.8904,12.0328,11.6758
"""RNF11""",10.8576,10.9745,11.5712,10.8805,10.9162,11.4222,11.1758,10.5655,10.3277,10.7961,11.301,11.1193,11.2544,10.652,10.9044,10.862,10.8794,11.3657,12.3537,11.0501,11.4506,11.2887,11.641,11.1991,10.6773,10.7971,10.2678,10.644,11.1081,11.8669,10.9774,11.8384,10.4421,11.5667,11.4618,11.3718,…,11.3674,11.2222,9.6044,11.3772,11.0512,11.1212,10.9594,11.3333,10.8986,10.6561,10.7207,11.3989,11.297,10.6685,10.9864,10.9277,11.603,11.332,11.0978,11.3011,11.125,11.1211,11.0442,10.8692,11.7259,11.1687,11.3986,9.7879,11.3008,10.7782,11.0804,10

In [26]:
# TODO, join genes with identical features into a single vertex
def find_identical_rows(matrix):
    # Convert the matrix to a structured array
    structured_array = np.core.records.fromarrays(matrix.transpose())
    
    # Find unique rows and their corresponding labels
    _, labels = np.unique(structured_array, return_inverse=True)
    
    # Group indices by labels
    unique_labels, indices = np.unique(labels, return_index=True)
    identical_rows_indices = [np.where(labels == label)[0] for label in unique_labels]
    
    return identical_rows_indices

# Example usage
matrix = np.array([[1, 2], [3, 4], [1, 2], [5, 6], [3, 4]])
identical_rows_indices = find_identical_rows(matrix)

for indices in identical_rows_indices:
    print(f"Identical rows at indices: {indices}")

# identical_rows = find_identical_rows(cna_m)

Identical rows at indices: [0 2]
Identical rows at indices: [1 4]
Identical rows at indices: [3]


In [5]:
# apply individual feature selection

X_mrna = mrna[:,1:].to_numpy()
X_cna = cna[:,1:].to_numpy()
X_mirna = mirna[:,1:].to_numpy()

mrna_mask, mrna_idx = variational_selection(X_mrna)
cna_mask, cna_idx = variational_selection(X_cna)
mirna_mask, mirna_idx = variational_selection(X_mirna)

X_mrna = X_mrna[:, mrna_mask]
X_cna = X_cna[:, cna_mask]
X_mirna = X_mirna[:, mirna_mask]

TypeError: variational_selection() missing 1 required positional argument: 'y'

In [ ]:
def get_mirna_gene_interactions(mirna_names, gene_names):
    """
    Retrieve interactions between mirnas and genes
    """
    ...


In [ ]:
# concat the features

In [ ]:
# run benchmarks